# DigiGestures: Gesture Classifier Training Pipeline

This notebook loads the hand landmark features collected from the webcam, trains a Random Forest model, visualizes performance metrics, and saves the trained classifier package.

In [ ]:
import pickle
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split

# Add src to Python path so we can import modules if needed
sys.path.append(str(Path("..").resolve()))

## 1. Load Dataset

We load from `data/hand_data.csv`. If the dataset has no samples, we run the automated synthetic data generator from our training package to create dummy samples for testing.

In [ ]:
from src.train import load_dataset, PROJECT_ROOT, MODEL_PATH, DATA_PATH, LABEL_TO_ID, ID_TO_LABEL, TOTAL_FEATURES

df = load_dataset(DATA_PATH)
print(f"Dataset shape: {df.shape}")
print(f"Target class counts:\n{df['label'].map(ID_TO_LABEL).value_counts()}")

## 2. Train / Test Split

In [ ]:
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Testing set: {X_test.shape[0]} samples")

## 3. Train Classifier

In [ ]:
print("Training Random Forest Classifier...")
clf = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42)
clf.fit(X_train, y_train)
print("Model training complete.")

## 4. Evaluate Performance

We calculate prediction accuracy, print the classification report, and display a visual confusion matrix.

In [ ]:
y_pred = clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {accuracy * 100:.2f}%\n")

unique_labels = np.unique(np.concatenate([y_test, y_pred]))
target_names = [ID_TO_LABEL[lid] for lid in unique_labels]

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=target_names))

# Plot confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm, 
    annot=True, 
    fmt="d", 
    cmap="Blues", 
    xticklabels=target_names, 
    yticklabels=target_names
)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

## 5. Save Model Package

In [ ]:
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

model_package = {
    "classifier": clf,
    "label_to_id": LABEL_TO_ID,
    "id_to_label": ID_TO_LABEL,
    "feature_count": TOTAL_FEATURES
}

with MODEL_PATH.open("wb") as file:
    pickle.dump(model_package, file)
    
print(f"Model package successfully saved to {MODEL_PATH}")